## Transformer Architecture — Step-by-Step Implementation

This notebook implements a small **Encoder–Decoder Transformer from basic NumPy operations** to make the internal data flow visible. Instead of using ready-made Transformer modules, each main operation is implemented separately using simple matrices.

A small vocabulary and two example sentences are used throughout:

- **Encoder input:** `I like cats`
- **Decoder input:** `You love dogs`

The notebook follows the complete data path:

**Text → Tokenization → One-Hot (Token) Encoding → Embedding → Q/K/V → Self-Attention → Multi-Head Attention → Residual + LayerNorm → FFN → Encoder → Masked Decoder Attention → Cross-Attention → Vocabulary Projection → Token Prediction**

The encoder converts input tokens into **context-aware representations**. The decoder uses its previous tokens together with the encoder output to produce a probability distribution over the vocabulary and predict the next token.

All matrices are intentionally very small (`d_model = 2`) so their operations and dimensions can be followed manually. The current weights are manually defined rather than trained, so the generated tokens are only for understanding the architecture, not meaningful language predictions.

> **Note:** This notebook currently focuses on the Transformer forward data flow. Training, loss calculation, backpropagation, learned weights, and positional encoding are not yet implemented.

<table>
<tr>
<td><img src="asset/images/sup-1.png" width="500"></td>
<td><img src="asset/images/sup-2.png" width="500"></td>
<td><img src="asset/images/sup-3.png" width="500"></td>
</tr>
</table>
<table>
<tr>
<td><img src="asset/images/sup-4.png" width="400"></td>
<td><img src="asset/images/sup-5.png" width="400"></td>
</tr>
</table>

### Setup and Parameters
Defines the small dataset, embedding dimension, attention weight matrices, FFN parameters, and cross-attention parameters used throughout the educational Transformer.

In [132]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, get_dataset_config_names
dataset_name = "GEM/wiki_auto_asset_turk"

train_data = load_dataset(
    dataset_name,
    "wiki_auto_asset_turk",
    split="train[:5000]"
)

print(train_data)

print(train_data.column_names)

print(train_data[0])

# Use "true" for the educational loop implementation; any other value uses NumPy's @ operator.
modif_matrix_multiplication = "true"

sentences = [
    "I like cats",
    "I like dogs",
    "You like cats",
    "You love dogs"
]

# Select one input sentence
sentence = "I like cats"
d = 2
W_Q = np.array([
    [1.0, 0.0],
    [0.0, 1.0]
])

W_K = np.array([
    [0.8, 0.2],
    [0.1, 0.9]
])

W_V = np.array([
    [0.7, 0.3],
    [0.4, 0.6]
])
#FFN Parameters
d_model = 2
d_ff = 4
# First linear layer: 2 -> 4
W_1 = np.array([
    [0.5, 0.2],
    [0.3, 0.7],
    [0.6, 0.1],
    [0.4, 0.8]
])
b_1 = np.array([
    [0.1],
    [0.1],
    [0.1],
    [0.1]
])
# Second linear layer: 4 -> 2
W_2 = np.array([
    [0.4, 0.2, 0.5, 0.3],
    [0.1, 0.6, 0.2, 0.7]
])
b_2 = np.array([
    [0.1],
    [0.1]
])


# --------------------------------------------------
# Encoder-Decoder Attention
# Cross-Attention
# --------------------------------------------------

W_Q_cross = np.array([
    [1.0, 0.0],
    [0.0, 1.0]
])

W_K_cross = np.array([
    [0.8, 0.2],
    [0.1, 0.9]
])

W_V_cross = np.array([
    [0.7, 0.3],
    [0.4, 0.6]
])



wiki_auto_asset_turk/train-00000-of-0000(…): reconstructing file:   0%|          |  0.00B / 89.5MB            

wiki_auto_asset_turk/train-00000-of-0000(…): downloading bytes:           |  0.00B            

wiki_auto_asset_turk/validation-00000-of(…): reconstructing file:   0%|          |  0.00B / 1.91MB            

wiki_auto_asset_turk/validation-00000-of(…): downloading bytes:           |  0.00B            

wiki_auto_asset_turk/test_asset-00000-of(…): reconstructing file:   0%|          |  0.00B /  204kB            

...

Generating challenge_test_turk_backtranslation split:   0%|          | 0/359 [00:00<?, ? examples/s]

Generating challenge_test_turk_bfp02 split:   0%|          | 0/359 [00:00<?, ? examples/s]

Generating challenge_test_turk_bfp05 split:   0%|          | 0/359 [00:00<?, ? examples/s]

Generating challenge_test_turk_nopunc split:   0%|          | 0/359 [00:00<?, ? examples/s]

Dataset({
    features: ['gem_id', 'gem_parent_id', 'source', 'target', 'references'],
    num_rows: 5000
})
['gem_id', 'gem_parent_id', 'source', 'target', 'references']
{'gem_id': 'wiki_auto_asset_turk-train-0', 'gem_parent_id': 'wiki_auto_asset_turk-train-0', 'source': 'Pterocarpus indicus ( commonly known as Amboyna wood , Malay padauk , Papua New Guinea rosewood , Philippine mahogany , Andaman redwood , Burmese rosewood , narra , angsana , or Pashu padauk ) is a species of " Pterocarpus " native to southeastern Asia , northern Australasia , and the western Pacific Ocean islands , in Cambodia , southernmost China , East Timor , Indonesia , Malaysia , Papua New Guinea , the Philippines , the Ryukyu Islands , the Solomon Islands , Thailand , and Vietnam .', 'target': 'Pterocarpus indicus ( commonly known as Amboyna wood , Malay padauk , Papua New Guinea rosewood , Philippine mahogany , Andaman redwood , Burmese rosewood , narra or Pashu padauk ) is a species of " Pterocarpus " native

#### img

<img src="asset/images/1.png" style="width:50%;">

### Helper Functions
Defines reusable operations such as Softmax, Layer Normalization, input preparation, cross-attention, and one complete encoder block.

In [119]:
# --------------------------------------------------
# Softmax
# --------------------------------------------------
def softmax(x):
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# --------------------------------------------------
# Layer Normalization
# --------------------------------------------------
def layer_norm(x, eps=1e-5):
    mean = np.mean(x, axis=0, keepdims=True)
    variance = np.var(x, axis=0, keepdims=True)
    normalized = (x - mean) / np.sqrt(variance + eps)
    return normalized

# --------------------------------------------------
# Rokinization and Embedded Token
# --------------------------------------------------
def prepare_input(sentence, token_to_id, vocab_size, E):

    tokens = sentence.split()

    token_ids = [
        token_to_id[token]
        for token in tokens
    ]

    one_hot_vectors = []

    for token_id in token_ids:

        one_hot = np.zeros((vocab_size, 1))

        one_hot[token_id, 0] = 1

        one_hot_vectors.append(one_hot)

    embedded_tokens = []

    for one_hot in one_hot_vectors:

        # embedded_token = E @ one_hot
        embedded_token = matrix_multiply(E,one_hot)

        embedded_tokens.append(embedded_token)

    X = np.hstack(embedded_tokens)

    return tokens, token_ids, X

# --------------------------------------------------
# Encoder-Decoder Attention
# Cross-Attention
# --------------------------------------------------
def cross_attention(decoder_attention_output,encoder_output):
    # Query comes from decoder
    Q_cross = matrix_multiply(W_Q_cross, decoder_attention_output)

    # Key and Value come from encoder
    K_cross = matrix_multiply(W_K_cross, encoder_output)
    V_cross = matrix_multiply(W_V_cross, encoder_output)

    d_k = Q_cross.shape[0]
    cross_scores = (matrix_multiply(Q_cross.T, K_cross))/ np.sqrt(d_k)
    cross_attention_weights = softmax(cross_scores)
    cross_attention_output = matrix_multiply(V_cross, cross_attention_weights.T)
    # print(cross_attention_output)
    return cross_attention_output

# --------------------------------------------------
# One Encoder Block
# --------------------------------------------------
def encoder_block(X, W_Q, W_K, W_V, W_O, W_1, b_1, W_2, b_2):
    # Self-attention
    Q = matrix_multiply(W_Q, X)
    K = matrix_multiply(W_K, X)
    V = matrix_multiply(W_V, X)

    d_k = Q.shape[0]
    scores = (matrix_multiply(Q.T, K)) / np.sqrt(d_k) #A
    attention_weights = softmax(scores) #A'
    attention = matrix_multiply( V , attention_weights.T)
    attention_output = matrix_multiply(W_O , attention)

    # First Residual + LayerNorm
    norm1_output = layer_norm(
        X + attention_output
    )

    # Feed-forward network
    hidden = matrix_multiply( W_1 , norm1_output) + b_1
    hidden = np.maximum(0, hidden)
    ffn_output = matrix_multiply( W_2 , hidden) + b_2
    
    # Second Residual + LayerNorm
    encoder_output = layer_norm(
        norm1_output + ffn_output
    )

    return encoder_output

def matrix_multiply(A, B):

    if modif_matrix_multiplication != "true":
        return A @ B

    rows_A = A.shape[0]
    cols_A = A.shape[1]

    rows_B = B.shape[0]
    cols_B = B.shape[1]

    result = np.zeros((rows_A, cols_B))

    for i in range(rows_A):
        for j in range(cols_B):
            for k in range(cols_A):
                tmp = A[i, k] * B[k, j]
                result[i, j] = tmp + result[i, j]

    return result

### Tokenization and Embedding
Splits text into tokens, assigns each token an ID, converts it to a one-hot vector, and multiplies it by the embedding matrix to obtain a dense token representation.
$x_{embedded}=Ex_{one−hot}$

A major part of this code also can be replaced by calling "prepare_input" function.
	​


In [133]:
# 2. Tokenization
tokenized_sentences = [
    sentence.split()
    for sentence in sentences
]

# 3. Vocabulary
all_tokens = [
    token
    for sentence in tokenized_sentences
    for token in sentence
]
vocabulary = sorted(set(all_tokens))

# 4. Token -> ID
token_to_id = {
    token: token_id
    for token_id, token in enumerate(vocabulary)
}

id_to_token = {
    token_id: token
    for token, token_id in token_to_id.items()
}

# Selected Sentence
tokens = sentence.split()
token_ids = [
    token_to_id[token]
    for token in tokens
]


vocab_size = len(vocabulary)
one_hot_vectors = []
for token_id in token_ids:
    one_hot = np.zeros((vocab_size, 1))
    one_hot[token_id, 0] = 1
    one_hot_vectors.append(one_hot)



# 6. Embedding matrix

E = np.array([
    [0.2, 0.3, 0.9, 0.8, 0.4, 0.5],
    [0.8, 0.7, 0.1, 0.2, 0.6, 0.7]
])


# 7. Embed each token
embedded_tokens = []
for one_hot in one_hot_vectors:
    embedded_token = matrix_multiply(E , one_hot)
    
    embedded_tokens.append(embedded_token)
X = np.hstack(embedded_tokens)

########### Alternative: Sof far can be replace with ##############

# encoder_tokens, encoder_ids, encoder_X = prepare_input(
#     "I like cats",
#     token_to_id,
#     vocab_size,
#     E
# )

######################################

#### img2 
<img src="asset/images/2.png" style="width:50%;">

### Query, Key, Value and Self-Attention

Each embedded token is projected into three different representations:

$Q=W_QX,\qquad K=W_KX,\qquad V=W_VX$

`Q` represents what a token looks for, `K` represents what it can match with, and `V` contains the information that can be transferred.


Queries and Keys are compared to calculate how strongly each token should attend to other tokens.

$A=\frac{Q^TK}{\sqrt{d_k}}$

Softmax converts these scores into attention weights, which are used to combine the Value vectors.

$Z=V\,Softmax(A)^T$

In [121]:
Q = matrix_multiply(W_Q, X)
K = matrix_multiply(W_K, X)
V = matrix_multiply(W_V, X)

# Self-Attention Mechanism: Attention (Q,K,V)= V.[Softmax(Kt.Q / (D_{k}^1/2))]t
A = matrix_multiply(Q.T , K)
d_k = Q.shape[0]
scores = A / np.sqrt(d_k)
attention_weights = softmax(scores) # A'
# Updated token representations
attention_output = matrix_multiply(V , attention_weights.T)

print(attention_output)

[[0.48761257 0.49418955 0.51150268]
 [0.50619372 0.50290522 0.49424866]]


#### img2 
<img src="asset/images/3.png" style="width:50%;">
<img src="asset/images/4.png" style="width:50%;">

### Multi-Head Attention

Multiple attention heads perform self-attention independently using different Q, K, and V projections. Their outputs are concatenated and projected through $W_O$, allowing different heads to capture different relationships between tokens.

In [122]:
# --------------------------------------------------
# Multi-Head Attention
# 2 heads
# each head dimension = 1
# --------------------------------------------------
num_heads = 2
d_head = 1

W_Q1 = np.array([
    [1.0, 0.0]
])
W_K1 = np.array([
    [0.8, 0.2]
])
W_V1 = np.array([
    [0.7, 0.3]
])

W_Q2 = np.array([
    [0.0, 1.0]
])
W_K2 = np.array([
    [0.1, 0.9]
])
W_V2 = np.array([
    [0.4, 0.6]
])

# Output Projection
W_O = np.array([
    [0.6, 0.4],
    [0.3, 0.7]
])

In [123]:
# Head 1

Q1 = matrix_multiply(W_Q1, X)
K1 = matrix_multiply(W_K1, X)
V1 = matrix_multiply(W_V1, X)
scores1 = matrix_multiply(Q1.T, K1) / np.sqrt(d_head)
attention_weights1 = softmax(scores1)
head1_output = matrix_multiply(V1 ,attention_weights1.T)

# Head 2
Q2 = matrix_multiply(W_Q2, X)
K2 = matrix_multiply(W_K2, X)
V2 = matrix_multiply(W_V2, X)
scores2 = matrix_multiply(Q2.T, K2) / np.sqrt(d_head)
attention_weights2 = softmax(scores2)
head2_output = matrix_multiply(V2 ,attention_weights2.T)

# Concatenate heads
multi_head_output = np.vstack([
    head1_output,
    head2_output
])

# Output projection
attention_output = matrix_multiply(W_O , multi_head_output)


#### img4
<img src="asset/images/5.png" style="width:50%;">

### Residual Connection and Layer Normalization

The original token representation is added to the attention output before normalization.
$X_{norm}=LayerNorm(X+Attention(X))$

The residual connection preserves previous information, while LayerNorm stabilizes the representation.

In [124]:
# --------------------------------------------------
# Residual connection + LayerNorm after attention
# --------------------------------------------------
attention_residual = X + attention_output
norm1_output = layer_norm(attention_residual)

### Feed-Forward Network
Each token is independently passed through the same two-layer neural network.

$FFN(x)=W_2\,ReLU(W_1x+b_1)+b_2$

The first layer expands from `d_model` to `d_ff`, and the second layer returns it to `d_model`.

In [130]:
# --------------------------------------------------
# FFN
# --------------------------------------------------
# First linear transformation

#####################  IF ###########################
# if LAyerNorm + Residual, then:
hidden = matrix_multiply(W_1 , norm1_output) + b_1

# else:
# hidden = matrix_multiply(W_1 , attention_output) + b_1
#####################################################

# ReLU activation
hidden = np.maximum(0, hidden)

# Second linear transformation
ffn_output = matrix_multiply(W_2 , hidden) + b_2

#####################  IF ###########################
# if LAyerNorm + Residual after FFN, then:
ffn_residual = norm1_output + ffn_output
encoder_output = layer_norm(ffn_residual)
print(encoder_output)
#####################################################

[[-0.99999653 -0.99999652  0.99999622]
 [ 0.99999653  0.99999652 -0.99999622]]


#### img5
<img src="asset/images/6.png" style="width:50%;">

### Stacked Encoder Blocks
The output of one encoder block becomes the input of the next encoder block. Each block progressively creates a more contextual representation of every token.

In [126]:
encoder_1_output = encoder_block(X,W_Q,W_K,W_V,W_O,
                                 W_1,b_1,
                                 W_2,b_2)

print(encoder_1_output)
encoder_2_output = encoder_block(encoder_1_output,W_Q,W_K,W_V,W_O,
                                 W_1,b_1,
                                 W_2,b_2)
print(encoder_2_output)

[[-0.99999653 -0.99999652  0.99999622]
 [ 0.99999653  0.99999652 -0.99999622]]
[[-0.99999653 -0.99999653  0.99999622]
 [ 0.99999653  0.99999653 -0.99999622]]


### Decoder Input and Masked Self-Attention

The decoder processes its input tokens similarly to the encoder, but applies a causal mask. The mask prevents each position from seeing future tokens.

$Attention(Q,K,V)=Softmax\left(\frac{Q^TK+Mask}{\sqrt{d_k}}\right)V$

In [127]:
# --------------------------------------------------
# Decoder
# --------------------------------------------------
decoder_sentence = "You love dogs"
decoder_tokens, decoder_ids, decoder_X = prepare_input(
    decoder_sentence,
    token_to_id,
    vocab_size,
    E
)

# --------------------------------------------------
# Decoder Q, K, V
# --------------------------------------------------

Q_dec = matrix_multiply(W_Q, decoder_X)
K_dec = matrix_multiply(W_K, decoder_X)
V_dec = matrix_multiply(W_V, decoder_X)

d_k = Q_dec.shape[0]
decoder_scores = matrix_multiply(Q_dec.T , K_dec) / np.sqrt(d_k)

sequence_length = decoder_scores.shape[0]

# Masked Self-Attention
mask = np.triu(np.ones((sequence_length, sequence_length)),k=1)
masked_scores = np.where(mask == 1,-np.inf,decoder_scores)
decoder_attention_weights = softmax(masked_scores)
decoder_attention_output = matrix_multiply(V_dec , decoder_attention_weights.T)
# print(decoder_attention_output)

#### img6
<img src="asset/images/7.png" style="width:50%;">

### Encoder–Decoder Cross-Attention + Decoder Feed-Forward and Output

Cross-attention connects the decoder to the encoder. The Query comes from the decoder, while the Keys and Values come from the encoder.
$Q=W_QX_{decoder}$

$K=W_KX_{encoder},\qquad V=W_VX_{encoder}$

This allows the decoder to retrieve relevant information from the encoded input.

After cross-attention, the decoder applies another residual connection, LayerNorm, and FFN. The resulting vectors are the final representations produced by the decoder block.

In [128]:
# 1. Residual + LayerNorm after Masked Self-Attention
decoder_residual_1 = decoder_X + decoder_attention_output
decoder_norm_1 = layer_norm(decoder_residual_1)
print(decoder_norm_1)
print(encoder_output)
# 2. Encoder-Decoder Cross-Attention
cross_attention_output = cross_attention(decoder_norm_1,encoder_output)

# 3. Residual + LayerNorm after Cross-Attention
decoder_residual_2 = (decoder_norm_1 + cross_attention_output)
decoder_norm_2 = layer_norm(decoder_residual_2)

# 4. Decoder Feed-Forward Network

decoder_hidden = matrix_multiply( W_1 , decoder_norm_2) + b_1
#Relu
decoder_hidden = np.maximum(0,decoder_hidden)
decoder_ffn_output = matrix_multiply( W_2 , decoder_hidden) + b_2
# 5. Residual + LayerNorm
decoder_residual_3 = (decoder_norm_2 + decoder_ffn_output)
decoder_output = layer_norm(decoder_residual_3)

[[-0.99992604 -0.99976063  0.99994534]
 [ 0.99992604  0.99976063 -0.99994534]]
[[-0.99999653 -0.99999652  0.99999622]
 [ 0.99999653  0.99999652 -0.99999622]]


### Token Generation

The decoder representation is projected from `d_model` to `vocab_size`, producing one score (logit) for every possible token.
$logits=W_{vocab}X_{decoder}+b_{vocab}$

Softmax converts the logits into token probabilities, and the token with the highest probability can be selected as the prediction.

In [129]:
# --------------------------------------------------
# Genration
# --------------------------------------------------

W_vocab = np.array([
    [0.2, 0.8],   # I
    [0.4, 0.5],   # You
    [0.9, 0.1],   # cats
    [0.8, 0.3],   # dogs
    [0.3, 0.7],   # like
    [0.5, 0.6]    # love
])

b_vocab = np.zeros((vocab_size, 1))

logits = matrix_multiply(W_vocab , decoder_output) + b_vocab

probabilities = softmax(logits)
predicted_ids = np.argmax(
    probabilities,
    axis=0
)
predicted_tokens = [
    id_to_token[token_id]
    for token_id in predicted_ids
]
print(predicted_tokens)

['I', 'I', 'cats']


### img7
<img src="asset/images/8.png" style="width:50%;">